In [0]:
%run ./Common_utils

In [0]:
%run ./api_config

In [0]:
%run ./issues_config

In [0]:
%run ./Jira_Client

In [0]:
import json
from concurrent.futures import ThreadPoolExecutor

In [0]:
with open("jira_endpoints.json", "r") as f:
    config = json.load(f)

In [0]:
issues_config = config["issues"]
projects_config = config["projects"]

In [0]:
JIRA_USER = ""
JIRA_API_TOKEN = 
auth = (JIRA_USER, JIRA_API_TOKEN)
headers = {"Accept": "application/json"}

In [0]:
TARGET_DATABASE = "Data_ingestion2"
BRONZE_TABLE = "issues_bronze2"
FLAT_TABLE = "issues_flat2"
RUN_LOG_TABLE = "pipeline_run_log2"

In [0]:
client = JiraClient(base,version,auth,headers)

In [0]:
projects = client.fetch(
    endpoint=projects_config["endpoint"],
    params ={},
    data_key=projects_config["data_key"]
)

project_keys = [p["key"] for p in projects if p.get("key")]  # Extracts the project key ("key") from each dictionary of proj from above returned list.


In [0]:
def fetch_issues_for_project(key):
    jql ='fproject ="{key}"'

    issues = client.fetch(
        endpoint=issues_config["endpoint"],
        params = {"jql": jql,
                 "fields": issues_config["fields"]
                 },
        data_key = issues_config["data_key"]

    )
    return apply_parser(issues, parse_issue)

In [0]:
all_issues_raw = []

with ThreadPoolExecutor(max_workers=5)as executor :
    results = executor.map(fetch_issues_for_project, project_keys)

for result in results :
    all_issues_raw.extend(result)


In [0]:
df_raw = create_df(spark, all_issues_raw) # Call "create_df" from common_utils

df_bronze = (
    df_raw
    .withColumn("parsed", F.from_json("raw_json", jira_schema))
)

df_flat = df_bronze.select(
    F.col("parsed.id").alias("ID"),
    F.col("parsed.key").alias("key"),
    F.col("parsed.fields.*")
)

In [0]:
display(df_raw)

In [0]:
display(df_bronze)

In [0]:
display(df_flat)

In [0]:
#spark.sql(f"CREATE DATABASE IF NOT EXISTS {TARGET_DATABASE}")
#write_table(df_flat, f"{TARGET_DATABASE}.{FLAT_TABLE}")

In [0]:
# log_pipeline_run(
#     spark,
#     f"{TARGET_DATABASE}.{RUN_LOG_TABLE}",
#     pipeline_name="issues",
#     status="SUCCESS",
#     records_processed=len(all_issues_raw),
#     projects_processed=len(project_keys)
# )